# Notebook 06: Random Forest Model & Feature Importance
## ML-Based Liquidity and Trading-Risk Intelligence for Indian Equities

**Phase:** Phase 8 Batch 4 (Stages 8.9–8.11)
**Target:** High-Risk Event (`hre_5d_train`, 5-day forward horizon)
**Evaluation Protocol:** 10-Fold Expanding-Window Walk-Forward Validation with 5-Session Purging

### Objectives
1. Train Random Forest Classifier across 10 expanding-window walk-forward folds.
2. Compare **Candidate A (35 features, baseline)** vs. **Candidate B (36 features, including F51 `relative_volume`)**.
3. Leverage scikit-learn 1.8.0 native missing-value routing during tree splitting (no imputation required).
4. Analyze the empirical impact of F51 (15.5% missingness) on validation PR-AUC, stability, and calibration.
5. Extract Gini feature importances to identify dominant liquidity and risk drivers.


In [1]:
# Imports and Environment Setup
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add repository root to path regardless of working directory
current_dir = Path('.').resolve()
if (current_dir / 'data' / 'processed').exists():
    REPO_ROOT = current_dir
elif (current_dir.parent / 'data' / 'processed').exists():
    REPO_ROOT = current_dir.parent
else:
    REPO_ROOT = current_dir
    for p in [current_dir] + list(current_dir.parents):
        if (p / 'data' / 'processed').exists():
            REPO_ROOT = p
            break

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.modeling.preprocessing import build_random_forest_pipeline, get_model_features
from src.modeling.evaluation import ModelEvaluator

print(f'Repository root: {REPO_ROOT}')
print('Environment initialized successfully.')


Repository root: C:\Users\vighn\Downloads\Indian_Equities_Liquidity_Risk_ML-main\Indian_Equities_Liquidity_Risk_ML-main
Environment initialized successfully.


### 1. Candidate Feature Sets & Native NaN Handling

- **Candidate A (35 features):** Approved baseline feature set excluding F51 `relative_volume`.
- **Candidate B (36 features):** Includes F51 `relative_volume` to test if tree splits exploit volume shocks despite 15.5% NaN.
- Both configurations use `class_weight='balanced_subsample'`, `max_depth=8`, `min_samples_leaf=50`, `n_estimators=150`.


In [2]:
features_a = get_model_features('random_forest', candidate='A')
features_b = get_model_features('random_forest', candidate='B')

print(f'Candidate A Feature Count: {len(features_a)}')
print(f'Candidate B Feature Count: {len(features_b)}')
diff = set(features_b) - set(features_a)
print(f'Difference (Candidate B unique): {diff}')

pipe_rf_a = build_random_forest_pipeline(candidate='A')
print('\nPipeline structure (no imputer step; native NaN handling):', pipe_rf_a)


Candidate A Feature Count: 35
Candidate B Feature Count: 36
Difference (Candidate B unique): {'relative_volume'}

Pipeline structure (no imputer step; native NaN handling): Pipeline(steps=[('classifier',
                 RandomForestClassifier(class_weight='balanced_subsample',
                                        max_depth=8, min_samples_leaf=50,
                                        n_estimators=150, n_jobs=-1,
                                        random_state=42))])


### 2. Walk-Forward Validation: Candidate A vs. Candidate B

Load the 10-fold walk-forward validation results and compare Candidate A vs. Candidate B fold-by-fold.


In [3]:
metrics_path = REPO_ROOT / 'data' / 'interim' / 'phase8_batch4' / 'supervised_fold_metrics.csv'
fold_metrics = pd.read_csv(metrics_path)

rf_metrics = fold_metrics[fold_metrics['model_name'].isin(['random_forest_candidate_a', 'random_forest_candidate_b'])]
display_cols = [
    'model_name', 'fold', 'val_start', 'val_end', 'val_rows',
    'pr_auc', 'roc_auc', 'brier_score', 'f1_50', 'calibrated_threshold', 'f1_calib'
]
rf_metrics[display_cols].sort_values(['model_name', 'fold'])


,model_name,fold,val_start,val_end,val_rows,pr_auc,roc_auc,brier_score,f1_50,calibrated_threshold,f1_calib
2,random_forest_candidate_a,0,2023-05-10,2023-08-02,15780,0.638079,0.859187,0.106765,0.556580,0.48,0.568731
7,random_forest_candidate_a,1,2023-08-03,2023-10-31,15821,0.600688,0.850836,0.124253,0.570315,0.38,0.545899
12,random_forest_candidate_a,2,2023-11-01,2024-01-30,15939,0.631150,0.849032,0.121231,0.590709,0.46,0.586819
17,random_forest_candidate_a,3,2024-01-31,2024-04-30,16157,0.581801,0.839496,0.151615,0.520572,0.48,0.509639
22,random_forest_candidate_a,4,2024-05-02,2024-07-29,16260,0.575204,0.838508,0.132540,0.546315,0.60,0.515317
27,random_forest_candidate_a,5,2024-07-30,2024-10-23,16358,0.600565,0.834356,0.129371,0.540524,0.52,0.539179
32,random_forest_candidate_a,6,2024-10-24,2025-01-20,16435,0.568349,0.835173,0.138395,0.538254,0.50,0.538254
37,random_forest_candidate_a,7,2025-01-21,2025-04-21,16685,0.563220,0.827521,0.172975,0.493631,0.54,0.508812
42,random_forest_candidate_a,8,2025-04-22,2025-07-15,16911,0.530808,0.803470,0.125956,0.504280,0.62,0.393693
47,random_forest_candidate_a,9,2025-07-16,2025-10-10,17011,0.524303,0.808657,0.125423,0.477898,0.46,0.499514


### 3. Aggregated Performance Comparison & F51 Impact

Compare mean and standard deviation of PR-AUC, ROC-AUC, Brier score, and calibrated F1.


In [4]:
rf_summary = rf_metrics.groupby('model_name').agg({
    'pr_auc': ['mean', 'std', 'min', 'max'],
    'roc_auc': ['mean', 'std'],
    'brier_score': ['mean'],
    'f1_50': ['mean'],
    'f1_calib': ['mean'],
    'calibrated_threshold': ['mean'],
    'fit_time_sec': ['sum', 'mean']
}).round(4)
rf_summary


pr_auc                         roc_auc          \
                             mean     std     min     max    mean     std   
model_name                                                                  
random_forest_candidate_a  0.5814  0.0377  0.5243  0.6381  0.8346  0.0177   
random_forest_candidate_b  0.5829  0.0352  0.5291  0.6353  0.8361  0.0169   

                          brier_score   f1_50 f1_calib calibrated_threshold  \
                                 mean    mean     mean                 mean   
model_name                                                                    
random_forest_candidate_a      0.1329  0.5339   0.5206                0.504   
random_forest_candidate_b      0.1320  0.5375   0.5252                0.508   

                          fit_time_sec           
                                   sum     mean  
model_name                                       
random_forest_candidate_a     682.0025  68.2002  
random_forest_candidate_b     790.0457  79.0046

### 4. Market Capitalization Performance

Evaluate out-of-fold performance across large, mid, and small market cap segments.


In [5]:
preds_path = REPO_ROOT / 'data' / 'interim' / 'phase8_batch4' / 'supervised_predictions.parquet'
preds_df = pd.read_parquet(preds_path)

for model in ['random_forest_candidate_a', 'random_forest_candidate_b']:
    print(f'=== Market Cap Stratification: {model} ===')
    sub_df = preds_df.rename(columns={'y_true': 'hre_5d_train', f'prob_{model}': 'hre_prob'})
    cap_eval = ModelEvaluator.evaluate_by_cap_segment(
        sub_df, segment_col='cap_segment', target_col='hre_5d_train', prob_col='hre_prob', threshold=0.50
    )
    print(cap_eval.to_string(index=False))
    print()


=== Market Cap Stratification: random_forest_candidate_a ===


cap_segment   pr_auc  roc_auc  threshold       f1  precision   recall  balanced_accuracy  brier_score  log_loss    tp    fp     tn    fn  n_obs  n_pos  prevalence
      large 0.642214 0.872432        0.5 0.350837   0.835438 0.222041           0.606126     0.105764  0.339255  2315   456  46129  8111  57011  10426    0.182877
        mid 0.709220 0.904246        0.5 0.645095   0.573131 0.737727           0.806081     0.122289  0.400454 11015  8204  57133  3916  80268  14931    0.186014
      small 0.650667 0.888220        0.5 0.456437   0.299166 0.962331           0.744506     0.225833  0.638801  4343 10174  11321   170  26008   4513    0.173524
        ALL 0.559061 0.824775        0.5 0.532504   0.484099 0.591664           0.725249     0.133011  0.417050 17673 18834 114583 12197 163287  29870    0.182929

=== Market Cap Stratification: random_forest_candidate_b ===


cap_segment   pr_auc  roc_auc  threshold       f1  precision   recall  balanced_accuracy  brier_score  log_loss    tp    fp     tn    fn  n_obs  n_pos  prevalence
      large 0.656957 0.877083        0.5 0.356250   0.867483 0.224151           0.608244     0.104437  0.334032  2337   357  46228  8089  57011  10426    0.182877
        mid 0.712582 0.906113        0.5 0.649611   0.574943 0.746568           0.810218     0.120847  0.395688 11147  8241  57096  3784  80268  14931    0.186014
      small 0.652416 0.888985        0.5 0.455116   0.297884 0.963882           0.743444     0.227650  0.642565  4350 10253  11242   163  26008   4513    0.173524
        ALL 0.561034 0.826601        0.5 0.535918   0.486139 0.597054           0.727880     0.132129  0.413483 17834 18851 114566 12036 163287  29870    0.182929



### 5. Random Forest Gini Feature Importances

Extract Gini feature importances from a training fit of Candidate B to inspect feature rankings.


In [6]:
data_path = REPO_ROOT / 'data' / 'processed' / 'features_with_targets.parquet'
df_full = pd.read_parquet(data_path)
train_mask = (df_full['split'] == 'train') & (~df_full['is_warmup']) & (df_full['hre_5d_train'].notna())

X_rf = df_full.loc[train_mask, features_b]
y_rf = df_full.loc[train_mask, 'hre_5d_train']

rf_pipe = build_random_forest_pipeline(candidate='B')
rf_pipe.fit(X_rf, y_rf)

rf_clf = rf_pipe.named_steps['classifier']
importances = pd.DataFrame({
    'feature': features_b,
    'gini_importance': rf_clf.feature_importances_
}).sort_values('gini_importance', ascending=False)

print('Top 15 Most Important Features in Random Forest:')
print(importances.head(15).to_string(index=False))


Top 15 Most Important Features in Random Forest:
         feature  gini_importance
  cs_spread_ma10         0.195275
daily_range_ma20         0.163343
    log_turnover         0.159494
         vol_20d         0.087582
downside_vol_20d         0.076490
     amihud_ma20         0.071338
      log_volume         0.042351
    amihud_ratio         0.030738
       vix_close         0.030031
   nifty_vol_20d         0.028584
rolling_corr_60d         0.027979
     daily_range         0.022635
          vol_5d         0.018063
   cs_spread_est         0.008325
      vol_of_vol         0.006615
